# MoteDB 10 分钟上手：机器人的"记忆体"

[MoteDB](https://github.com/motedb/motedb) 是 Rust 写的嵌入式多模态数据库——**向量、时序、全文、BLOB 一个 SQL 引程搞定**，专为具身智能（机器人/AR/边缘设备）设计。本 notebook 用 `pip` 安装的真实发行包跑通它的核心能力闭环：

1. **写入**机器人帧数据（时间戳 + 任务 + 事件注记 + 8 维 embedding）
2. **连续视图**（v0.18）：分钟级时序聚合，`FIRST(v, ts)` / `LAST(v, ts)`
3. **向量 KNN**：`ORDER BY emb <-> '[…]'`（DiskANN 索引加速）
4. **混合检索**：BM25 全文 × 向量 KNN，RRF 融合，一次调用
5. **Arrow 互操作**：直接出 `pyarrow.Table` 喂训练管线
6. **单文件快照**：`EXPORT` → 只读部署（舰队分发）

> 全程无服务进程、无外部依赖。Colab 上约 1 分钟跑完。


In [ ]:
%pip install -q motedb-python pyarrow pandas


## ① 数据模型：一个表装下机器人的一帧

真实机器人数据是"结构化 + 向量 + 文本"混合的。MoteDB 里就是一张普通的 SQL 表：


In [ ]:
import random, time, shutil
import motedb

# 幂等清理：重复运行本 notebook 不会撞 "table already exists"
shutil.rmtree("robot_run.mote", ignore_errors=True)
for f in ("fleet_snapshot.mote",):
    __import__("os").path.exists(f) and __import__("os").remove(f)

print("motedb-python", motedb.__version__)
db = motedb.Database("robot_run.mote")

db.execute("CREATE TABLE frames (ts TIMESTAMP, task TEXT, note TEXT, emb VECTOR(8))")

random.seed(7)
base = 1794132000  # 2026-11-08T10:00:00Z
tasks = ["pick", "place", "carry"]
notes = {
    "pick":  ["gripper closed cleanly", "object slipped mid-pick"],
    "place": ["placed at target",       "offset too large"],
    "carry": ["path smooth",            "obstacle stop"],
}
rows = []
for i in range(300):
    sec = base + i * 17
    task = tasks[i % 3]
    note = notes[task][0] if i % 7 else notes[task][1]   # 每 7 帧一次异常注记
    proto = [float(tasks.index(task))] + [0.0] * 7       # 任务原型向量 + 噪声
    emb = [round(x + random.uniform(-0.25, 0.25), 4) for x in proto]
    rows.append((f"2026-11-08T10:{(sec // 60) % 60:02d}:{sec % 60:02d}", task, note, emb))

db.executemany("INSERT INTO frames VALUES (?, ?, ?, ?)", rows)
print("inserted:", db.query("SELECT COUNT(*) FROM frames")[1])



## ② 连续视图（v0.18）：分钟桶实时聚合

`CREATE CONTINUOUS VIEW` 定义一次，之后**当普通表查**——外层 `WHERE` / 聚合 / `ORDER BY` 随便组合。刷新是查询时惰性完成的（v1 = 全量重算，查询时点一致）。

`FIRST(v, ts)` / `LAST(v, ts)` 是 0.18 新增的两参数时序聚合：桶内时间最早/最晚的值。


In [ ]:
db.execute("""CREATE CONTINUOUS VIEW frames_1m AS
    SELECT TIME_BUCKET('1m', ts) AS minute, task,
           COUNT(*)        AS n,
           FIRST(note, ts) AS first_note,
           LAST(note, ts)  AS last_note
    FROM frames GROUP BY minute, task""")

df = db.query_pandas(
    "SELECT minute, task, n, first_note FROM frames_1m ORDER BY minute, task")
print("连续视图行数:", len(df))
df.head(8)


再加一张数值传感器表看经典用法——`AVG` + `FIRST/LAST` 一行 SQL：


In [ ]:
db.execute("CREATE TABLE sensor (ts TIMESTAMP, device TEXT, v FLOAT)")
srows = [(f"2026-11-08T10:{(base + i * 17) // 60 % 60:02d}:{(base + i * 17) % 60:02d}",
          "arm_a" if i % 2 else "arm_b", round(random.uniform(0, 10), 3))
         for i in range(120)]
db.executemany("INSERT INTO sensor VALUES (?, ?, ?)", srows)

db.execute("""CREATE CONTINUOUS VIEW sensor_1m AS
    SELECT TIME_BUCKET('1m', ts) AS minute, device,
           AVG(v) AS v_avg, FIRST(v, ts) AS v_first, LAST(v, ts) AS v_last
    FROM sensor GROUP BY minute, device""")
db.query_pandas("SELECT * FROM sensor_1m ORDER BY minute, device LIMIT 6")


## ③ 向量 KNN：找"最像某个任务原型"的帧

`emb <-> '[…]'` 是 L2 距离排序；建了 `VECTOR INDEX` 后走 DiskANN 图索引。下面查"最像 carry 原型（第 2 维=2）"的帧：


In [ ]:
db.execute("CREATE VECTOR INDEX f_emb ON frames(emb)")
time.sleep(1.5)  # 索引异步构建窗口

cols, hits = db.query(
    "SELECT task, note FROM frames ORDER BY emb <-> '[2.0,0,0,0,0,0,0,0]' LIMIT 3")
print("最像 carry 原型的帧:", hits)


## ④ 全文检索 + 混合检索

BM25 全文（jieba 中文分词内置）——"哪几帧发生了滑落？"：


In [ ]:
db.execute("CREATE TEXT INDEX f_note ON frames(note)")
time.sleep(1.5)

db.query_pandas(
    "SELECT task, note FROM frames WHERE MATCH(note) AGAINST('slipped') LIMIT 5")


**混合检索**是 MoteDB 的招牌：一次调用把 BM25 与向量 KNN 用 RRF 融合——"语义接近 carry、文本提到 slip"的帧排最前：


In [ ]:
hy = db.hybrid_search("f_note", "slipped", "f_emb",
                     [0.0, 0, 0, 0, 0, 0, 0, 0], 5)  # 文本"slipped" × pick 原型
for r in hy:
    print(r["task"], "|", r["note"])



## ⑤ Arrow 互操作：直接喂训练管线

`query_arrow()` 返回规范 `pyarrow.Table`（VECTOR 列 → `FixedSizeList<float32>`），pandas/DuckDB/训练代码零拷贝对接：


In [ ]:
tbl = db.query_arrow("SELECT task, note, emb FROM frames LIMIT 5")
print(tbl.schema)
tbl.to_pandas()


## ⑥ 单文件快照：采集 → 运输 → 部署

机器人端是高写入目录；`EXPORT` 一条语句打包成带 CRC 的单文件（可选 zstd）；目标设备 `read_only=True` 打开即查，所有写路径拒绝、多读者共存：


In [ ]:
stats = db.export_to("fleet_snapshot.mote", compressed=True)
print("快照:", stats["files"], "files,", round(stats["bytes"] / 1024), "KiB")

ro = motedb.Database("fleet_snapshot.mote", read_only=True)
print("只读端帧数:", ro.query("SELECT COUNT(*) FROM frames")[1])
try:
    ro.execute("INSERT INTO frames VALUES ('2026-11-08T12:00:00','pick','x','[0,0,0,0,0,0,0,0]')")
except Exception as e:
    print("只读容器拒绝写入 ✓:", str(e)[:60])


## 收工

一个 `pip install`，六段 SQL，跑完了"采集 → 聚合 → 检索 → 导出训练 → 舰队分发"的完整闭环。

- **源码**（Rust，MIT）: https://github.com/motedb/motedb
- **文档**: https://motedb.github.io · **docs.rs**: https://docs.rs/motedb
- **Rust / Node**: `cargo add motedb` · `npm install motedb`
- **想贡献？** 仓库里挂着 [good first issue](https://github.com/motedb/motedb/issues?q=label%3A%22good+first+issue%22)
- **能力清单**（每行对应实现与测试）: 仓库 `CAPABILITIES.md`
